## Multi-Representation Indexing

In [5]:
import uuid
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryStore
from langchain_classic.retrievers import MultiVectorRetriever

In [2]:
# Load the Raw Documents 

print("--- 1. Loading Documents ---")

loader1 = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader1.load()

loader2 = WebBaseLoader("https://lilianweng.github.io/posts/2024-02-05-human-data-quality/")
docs.extend(loader2.load())

print(f"Total documents loaded: {len(docs)}")
print(f"Type of documents: {type(docs[0])}\n")

--- 1. Loading Documents ---
Total documents loaded: 2
Type of documents: <class 'langchain_core.documents.base.Document'>



In [3]:
# Generate Summaries (The "Bait")

print("--- 2. Generating Summaries ---")

# We use a simple chain to ask the LLM to summarize the page content
chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("Summarize the following document concisely:\n\n{doc}")
    | ChatOpenAI(model="gpt-4o-mini", max_retries=0)
    | StrOutputParser()
)

# .batch() processes all documents in parallel
summaries = chain.batch(docs, {"max_concurrency": 5})

print(f"Generated {len(summaries)} summaries.")
print(f"Type of summary output: {type(summaries[0])}")
print(f"Preview of first summary: {summaries[0][:150]}...\n")

--- 2. Generating Summaries ---
Generated 2 summaries.
Type of summary output: <class 'langchain_core.messages.base.TextAccessor'>
Preview of first summary: The document by Lilian Weng explores the development of LLM (Large Language Model)-powered autonomous agents, outlining their essential components: pl...



In [6]:
# Setup the Databases and Retriever 

print("--- 3. Setting up Multi-Vector Retriever ---")

# 3a. Vectorstore (To hold the embeddings of the summaries)
vectorstore = Chroma(
    collection_name="summaries",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small")
)

# 3b. Docstore (To hold the raw, full-text parent documents)
# Note: Replaced InMemoryByteStore with InMemoryStore to handle Document objects directly
store = InMemoryStore()
id_key = "doc_id"

# 3c. The Retriever that links them together
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)

--- 3. Setting up Multi-Vector Retriever ---
